# Download các thư viện cần thiết

In [1]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [2]:
def read_parquet_user(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [3]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [4]:
def read_parquet_transaction(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [5]:
def split_and_save_parquet(df, num_files, output_dir, type):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"sale_pers.{type}_chunk_{i}.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

# Chuẩn bị dữ liệu

In [6]:
df_user = read_parquet_user("./preprocessed-dataset")
df_user.head()

customer_id,gender,province,membership,created_date,last_sync_date,install_app,install_datetime,user_age_days,days_since_install,days_since_last_sync
i32,str,str,str,date,date,str,date,f64,f64,f64
9110357,"""female""","""Hồ Chí Minh""","""Standard""",2025-08-23,null,"""In-Store""",2025-08-23,38.596563,38.917091,null
9112155,"""female""","""Hà Nội""","""Standard""",2025-08-23,null,"""SPE""",2025-08-19,38.284309,42.917091,null
9111698,"""female""","""Hà Nội""","""Standard""",2025-08-23,null,"""SPE""",2025-08-23,38.375748,38.917091,null
9110836,"""male""","""Bình Dương""","""Standard""",2025-08-23,null,"""In-Store""",2025-08-23,38.509642,38.917091,null
9110747,"""female""","""Cà Mau""","""Standard""",2025-08-23,null,"""In-Store""",2025-08-23,38.524602,38.917091,null


In [7]:
df_transaction = read_parquet_transaction("./preprocessed-dataset")
df_transaction.head()

item_id,price,quantity,customer_id,created_date,channel,payment,location,discount,list_price,category_l2,discount_rate
str,"decimal[38,4]",i32,i32,date,str,str,i32,"decimal[38,4]","decimal[38,4]",str,"decimal[38,4]"
"""0029130000021""",54400.0000,1,6672770,2024-05-26,"""In-Store""","""VietQR""",483,9600.0000,69000.0000,"""Bột ăn dặm""",0.2116
"""5506000000005""",112370.4735,1,3501018,2024-05-17,"""In-Store""","""VietQR""",854,6629.5265,119000.0000,"""Snack ăn dặm""",0.0557
"""0020250010001""",165000.0000,1,4899881,2024-05-26,"""In-Store""","""Tiền mặt""",376,20000.0000,185000.0000,"""Giặt xả cho bé""",0.1081
"""3047000000002""",43000.0000,1,6408731,2024-05-26,"""In-Store""","""VietQR""",161,0.0000,47000.0000,"""Dầu ăn & Gia vị""",0.0851
"""5140000000011""",270000.0000,1,5676745,2024-05-17,"""In-Store""","""VietQR""",412,0.0000,270000.0000,"""Bình sữa, phụ kiện""",0.0000


In [8]:
df_item = read_parquet_item("./preprocessed-dataset")
df_item.head()

item_id,price,category_l1,category_l2,brand,item_type,color,size,gender_target_final,description_final,age_group_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Từ 9M"""
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Con Cưng""","""Bộ quần áo""","""Không xác định""","""Không xác định""","""Bé Gái""","""Không xác định""","""Từ 36M"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""0-12M"""
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""﻿﻿Tã dán Merries size S 82 miế…","""[""Từ 4M"", ""3M-6M"", ""12-36M""]"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""﻿﻿﻿Bỉm tã quần Merries size M …","""12-36M"""


# Training Stage 1

In [9]:
import numpy as np
import pandas as pd
import polars as pl

from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GridSearchCV

from tqdm.auto import tqdm  # <- thêm dòng này


/datastore/uittogether/tools/miniconda3/envs/MABe/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Chuẩn bị transaction, sort thời gian, tách train/valid (2024)

In [10]:
# df_transaction: Polars DataFrame gốc

df_trx = (
    df_transaction
    .select(["customer_id", "item_id", "created_date"])
    .drop_nulls(["customer_id", "item_id", "created_date"])
    .with_columns(
        pl.col("created_date")
        .cast(pl.Datetime)              # chuyển Date/String → Datetime
        .alias("created_datetime")
    )
    .sort("created_datetime")          # sắp xếp thời gian tăng dần
)

print("Tổng số dòng transaction:", df_trx.height)
print(df_trx.head())
print(df_trx.dtypes)

Tổng số dòng transaction: 35729825
shape: (5, 4)
┌─────────────┬───────────────┬──────────────┬─────────────────────┐
│ customer_id ┆ item_id       ┆ created_date ┆ created_datetime    │
│ ---         ┆ ---           ┆ ---          ┆ ---                 │
│ i32         ┆ str           ┆ date         ┆ datetime[μs]        │
╞═════════════╪═══════════════╪══════════════╪═════════════════════╡
│ 1028293     ┆ 4048000000008 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
│ 512190      ┆ 1158000000007 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
│ 512190      ┆ 1606000000010 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
│ 512190      ┆ 1627000000005 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
│ 409015      ┆ 2803000000012 ┆ 2024-01-01   ┆ 2024-01-01 00:00:00 │
└─────────────┴───────────────┴──────────────┴─────────────────────┘
[Int32, String, Date, Datetime(time_unit='us', time_zone=None)]


In [11]:
# Chỉ lấy năm 2024
df_trx_2024 = df_trx.filter(
    pl.col("created_datetime").dt.year() == 2024
)

# Train = tháng 1 → 11/2024
df_train_pl = df_trx_2024.filter(
    pl.col("created_datetime").dt.month().is_between(1, 11, closed="both")
)

# Valid = tháng 12/2024
df_valid_pl = df_trx_2024.filter(
    pl.col("created_datetime").dt.month() == 12
)

print("Số dòng train (Polars):", df_train_pl.height)
print("Số dòng valid (Polars):", df_valid_pl.height)


Số dòng train (Polars): 32680632
Số dòng valid (Polars): 3049193


In [12]:
df_train = df_train_pl.to_pandas()
df_valid = df_valid_pl.to_pandas()

print("Số dòng train (pandas):", len(df_train))
print("Số dòng valid (pandas):", len(df_valid))
print(df_train.head())
print(df_valid.head())


Số dòng train (pandas): 32680632
Số dòng valid (pandas): 3049193
   customer_id        item_id created_date created_datetime
0      1028293  4048000000008   2024-01-01       2024-01-01
1       512190  1158000000007   2024-01-01       2024-01-01
2       512190  1606000000010   2024-01-01       2024-01-01
3       512190  1627000000005   2024-01-01       2024-01-01
4       409015  2803000000012   2024-01-01       2024-01-01
   customer_id        item_id created_date created_datetime
0      6981489  6382000000005   2024-12-01       2024-12-01
1      7092360  5427000000006   2024-12-01       2024-12-01
2      6274269  0007090000357   2024-12-01       2024-12-01
3      3192555  3436000000013   2024-12-01       2024-12-01
4      4177385  3953000000092   2024-12-01       2024-12-01


## Định nghĩa Estimator Stage 1 (Item–Item CF + Cosine + optional TF-IDF)

In [13]:
class ItemItemCFStage1(BaseEstimator):
    """
    Stage 1: Item-Item Collaborative Filtering với:
      - Ma trận user-item có trọng số (log_count + ưu tiên category/age_group)
      - TF-IDF weighting trên ma trận user-item
      - Cosine similarity trên vector item sau TF-IDF
      - Cold-start user -> top-k item phổ biến nhất

    df_train : pandas DataFrame, chứa transaction train
    df_valid : pandas DataFrame, chứa transaction valid
    df_item  : pandas DataFrame, chứa metadata item

    Cột bắt buộc:
      - df_train: [user_col, item_col], nên có "quantity" nếu muốn trọng số tốt hơn
      - df_item : [item_col, "category_l2", "age_group_final"]
    """

    def __init__(
        self,
        df_train,
        df_valid,
        df_item,
        user_col="customer_id",
        item_col="item_id",
        weight_type="log_count",      # "binary" | "count" | "log_count" | "rel_freq"
        alpha_cat=0.5,
        alpha_age=0.5,
        n_neighbors=50,
        k_eval=100,
        use_tqdm=True,
    ):
        # dữ liệu & cấu hình
        self.df_train = df_train
        self.df_valid = df_valid
        self.df_item = df_item

        self.user_col = user_col
        self.item_col = item_col

        # siêu tham số ma trận user-item
        self.weight_type = weight_type    # base: binary/count/log_count/rel_freq
        self.alpha_cat = alpha_cat        # độ mạnh ưu tiên category_l2
        self.alpha_age = alpha_age        # độ mạnh ưu tiên age_group_final

        # siêu tham số CF
        self.n_neighbors = n_neighbors
        self.k_eval = k_eval
        self.use_tqdm = use_tqdm

    # =========================
    # Helper: build ma trận user-item có trọng số + side info item
    # =========================
    def _build_user_item_matrix(self):
        """
        Tạo ma trận user-item (CSR) từ df_train với trọng số w_ui:

            base = log1p(sum_qty(u,i)) hoặc biến thể
            w_ui = base * (1 + alpha_cat * cat_pref + alpha_age * age_pref)

        Trong đó:
          - cat_pref = P(category_l2(i) | user u)
          - age_pref = P(age_group_final(i) | user u)
        """

        df = self.df_train.copy()

        # chỉ giữ cột cần thiết
        cols_needed = [self.user_col, self.item_col]
        if "quantity" in df.columns:
            cols_needed.append("quantity")
        df = df[cols_needed]

        # aggregate theo (user, item)
        if "quantity" in df.columns:
            agg = (
                df.groupby([self.user_col, self.item_col], as_index=False)
                .agg(
                    n_interactions=(self.item_col, "size"),
                    sum_qty=("quantity", "sum"),
                )
            )
        else:
            agg = (
                df.groupby([self.user_col, self.item_col], as_index=False)
                .agg(
                    n_interactions=(self.item_col, "size"),
                )
            )
            agg["sum_qty"] = agg["n_interactions"]

        # ===== join thêm thông tin item: category_l2, age_group_final =====
        item_cols = [self.item_col, "category_l2", "age_group_final"]
        df_item_small = self.df_item[item_cols].drop_duplicates(subset=[self.item_col])
        agg = agg.merge(df_item_small, on=self.item_col, how="left")

        # fill NA cho feature dùng trong groupby
        agg["category_l2"] = agg["category_l2"].fillna("__UNK_CAT2__")
        agg["age_group_final"] = agg["age_group_final"].fillna("__UNK_AGE__")

        # ===== tính base weight từ sum_qty =====
        base_raw = agg["sum_qty"].astype(float)

        if self.weight_type == "binary":
            base = np.ones_like(base_raw, dtype=np.float32)
        elif self.weight_type == "count":
            base = base_raw.values.astype(np.float32)
        elif self.weight_type == "log_count":
            base = np.log1p(base_raw.values).astype(np.float32)
        elif self.weight_type == "rel_freq":
            # freq tương đối trong lịch sử user: freq_ui / tổng freq user
            user_total = agg.groupby(self.user_col)["sum_qty"].transform("sum")
            base = (base_raw / user_total).values.astype(np.float32)
        else:
            raise ValueError(f"Unknown weight_type: {self.weight_type}")

        agg["base_weight"] = base

        # ===== tính cat_pref & age_pref trên sum_qty =====
        # tổng quantity theo user (mẫu số)
        agg["user_total_qty"] = agg.groupby(self.user_col)["sum_qty"].transform("sum")

        # tổng quantity theo (user, category_l2)
        agg["user_cat_l2_qty"] = agg.groupby(
            [self.user_col, "category_l2"]
        )["sum_qty"].transform("sum")

        # tổng quantity theo (user, age_group_final)
        agg["user_ageg_qty"] = agg.groupby(
            [self.user_col, "age_group_final"]
        )["sum_qty"].transform("sum")

        # preference = freq_ui_feature / tổng của user
        agg["cat_pref"] = agg["user_cat_l2_qty"] / agg["user_total_qty"]
        agg["age_pref"] = agg["user_ageg_qty"] / agg["user_total_qty"]

        # fill NA nếu có
        agg["cat_pref"] = agg["cat_pref"].fillna(0.0)
        agg["age_pref"] = agg["age_pref"].fillna(0.0)

        # ===== final weight w_ui =====
        factor = 1.0 + self.alpha_cat * agg["cat_pref"] + self.alpha_age * agg["age_pref"]
        w_ui = agg["base_weight"].values * factor.values.astype(np.float32)

        agg["value"] = w_ui.astype(np.float32)

        # ===== mã hóa user_id, item_id -> index =====
        user_cat = agg[self.user_col].astype("category")
        item_cat = agg[self.item_col].astype("category")

        self.user_index_to_id_ = list(user_cat.cat.categories)
        self.item_index_to_id_ = list(item_cat.cat.categories)

        self.user_id_to_index_ = {uid: idx for idx, uid in enumerate(self.user_index_to_id_)}
        self.item_id_to_index_ = {iid: idx for idx, iid in enumerate(self.item_index_to_id_)}

        user_codes = user_cat.cat.codes.to_numpy()
        item_codes = item_cat.cat.codes.to_numpy()
        data = agg["value"].to_numpy(dtype=np.float32)

        n_users = len(self.user_index_to_id_)
        n_items = len(self.item_index_to_id_)

        ui_matrix = csr_matrix(
            (data, (user_codes, item_codes)),
            shape=(n_users, n_items),
            dtype=np.float32
        )

        # lịch sử train cho từng user (set index item)
        self.user_history_ = {}
        for u_idx, i_idx in zip(user_codes, item_codes):
            self.user_history_.setdefault(u_idx, set()).add(i_idx)

        # độ phổ biến item (để fallback cold-start user)
        item_popularity = np.asarray(ui_matrix.sum(axis=0)).ravel()
        self.item_popularity_ = item_popularity
        self.popular_item_indices_ = np.argsort(-item_popularity)

        return ui_matrix

    # =========================
    # Helper: build hàng xóm item trên TF-IDF
    # =========================
    def _build_item_neighbors(self, ui_matrix):
        """
        Áp TF-IDF lên ma trận user-item rồi fit NearestNeighbors trên vector item.
        """

        self.tfidf_ = TfidfTransformer(
            norm="l2",
            use_idf=True,
            sublinear_tf=True,
        )
        ui_tfidf = self.tfidf_.fit_transform(ui_matrix)

        # Mỗi item = 1 vector = 1 dòng trong ma trận chuyển vị
        X_items = ui_tfidf.T

        self.nn_model_ = NearestNeighbors(
            n_neighbors=self.n_neighbors + 1,  # +1 để bao gồm chính nó
            metric="cosine",
            algorithm="brute",
            n_jobs=-1,
        )
        self.nn_model_.fit(X_items)

        distances, indices = self.nn_model_.kneighbors(X_items, return_distance=True)
        sims = 1.0 - distances  # cosine similarity

        # Bỏ neighbor thứ 0 (chính item)
        self.item_neighbors_ = indices[:, 1:]
        self.item_neighbor_sims_ = sims[:, 1:]

    # =========================
    # Helper: recommend cho 1 user-index (warm-start)
    # =========================
    def _recommend_for_user_index(self, user_idx, top_k=None):
        """
        Recommend cho 1 user warm-start (có history trong train).
        Trả về list index item nội bộ.
        """
        if top_k is None:
            top_k = self.k_eval

        history = self.user_history_.get(user_idx, None)
        if not history:
            return []

        candidate_scores = {}

        for item_i in history:
            neighbors = self.item_neighbors_[item_i]
            sims = self.item_neighbor_sims_[item_i]

            for nbr_idx, sim in zip(neighbors, sims):
                if nbr_idx in history:
                    continue
                candidate_scores[nbr_idx] = candidate_scores.get(nbr_idx, 0.0) + float(sim)

        if not candidate_scores:
            return []

        sorted_candidates = sorted(
            candidate_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
        top_items = [idx for idx, _ in sorted_candidates[:top_k]]
        return top_items

    # =========================
    # API sklearn
    # =========================
    def fit(self, X=None, y=None):
        """
        Build model từ df_train:
        - Xây ma trận user-item có trọng số (dùng item metadata)
        - Áp TF-IDF
        - Fit item-item NearestNeighbors
        """
        ui_matrix = self._build_user_item_matrix()
        self._build_item_neighbors(ui_matrix)
        return self

    def score(self, X=None, y=None):
        """
        Tính mean Recall@k_eval trên df_valid.

        - Warm-start user (có lịch sử train)  -> dùng item-item CF
        - Cold-start user (chỉ có ở valid)   -> fallback top-k item phổ biến nhất
        """
        if not hasattr(self, "user_history_"):
            raise RuntimeError("Model chưa fit. Hãy gọi fit() trước score().")

        if self.df_valid.empty:
            return 0.0

        df_val = self.df_valid[[self.user_col, self.item_col]].drop_duplicates().copy()

        # map item_id valid sang index item train
        item_cat_val = pd.Categorical(df_val[self.item_col], categories=self.item_index_to_id_)
        df_val["item_idx"] = item_cat_val.codes

        # bỏ item cold-start (chưa có trong train)
        df_val = df_val[df_val["item_idx"] != -1]

        if df_val.empty:
            return 0.0

        # ground truth: user_id -> set(item_idx) trong valid
        user_to_true_items = (
            df_val.groupby(self.user_col)["item_idx"]
            .apply(lambda s: set(s.to_list()))
            .to_dict()
        )

        recalls = []
        n_eval_users = 0

        iterator = user_to_true_items.items()
        if self.use_tqdm:
            iterator = tqdm(
                iterator,
                total=len(user_to_true_items),
                desc=f"Evaluating users (k={self.k_eval}, w={self.weight_type}, nn={self.n_neighbors})",
                leave=False,
            )

        for user_id, true_items in iterator:
            # warm vs cold
            if user_id in self.user_id_to_index_:
                user_idx = self.user_id_to_index_[user_id]
                rec_items = self._recommend_for_user_index(user_idx, top_k=self.k_eval)
            else:
                # cold-start user -> top-k item phổ biến nhất
                rec_items = list(self.popular_item_indices_[: self.k_eval])

            if not rec_items:
                recalls.append(0.0)
                n_eval_users += 1
                continue

            rec_set = set(rec_items)
            inter = len(rec_set & true_items)
            recall_u = inter / len(true_items)
            recalls.append(recall_u)
            n_eval_users += 1

        if n_eval_users == 0:
            return 0.0

        mean_recall = float(np.mean(recalls))
        return mean_recall

    # =========================
    # Convenience: recommend theo user_id gốc (online)
    # =========================
    def recommend_for_user_id(self, user_id, top_k=None):
        """
        Recommend top_k item_id cho một user_id (id gốc).
        - Warm-start: dùng CF
        - Cold-start: top-k item phổ biến nhất
        """
        if top_k is None:
            top_k = self.k_eval

        if user_id in self.user_id_to_index_:
            user_idx = self.user_id_to_index_[user_id]
            item_idx_list = self._recommend_for_user_index(user_idx, top_k=top_k)
        else:
            item_idx_list = list(self.popular_item_indices_[: top_k])

        item_ids = [self.item_index_to_id_[idx] for idx in item_idx_list]
        return item_ids

## GridSearchCV để tìm siêu tham số tốt nhất

In [14]:
df_item_pd = df_item.to_pandas()

In [15]:
X_dummy = np.zeros((1, 1))


base_estimator = ItemItemCFStage1(
    df_train=df_train,
    df_valid=df_valid,
    df_item=df_item_pd,
    user_col="customer_id",
    item_col="item_id",
    use_tqdm=True,
)

# param_grid = {
#     "weight_type": ["log_count", "rel_freq"],   # có thể thử thêm "count", "binary" nếu muốn
#     "n_neighbors": [20, 50, 100],
#     "k_eval": [50, 100, 200],
#     # bạn cũng có thể thử tune alpha_cat, alpha_age nhưng nên để sau
# }

param_grid = {
    "weight_type": ["rel_freq"],   # có thể thử thêm "count", "binary" nếu muốn
    "n_neighbors": [100],
    "k_eval": [1000],
    # bạn cũng có thể thử tune alpha_cat, alpha_age nhưng nên để sau
}
cv = [(np.arange(len(X_dummy)), np.arange(len(X_dummy)))]

grid = GridSearchCV(
    estimator=base_estimator,
    param_grid=param_grid,
    cv=cv,
    scoring=None,
    refit=True,
    verbose=2,
    n_jobs=1,
)

grid.fit(X_dummy)

print("Best params:", grid.best_params_)
print("Best mean Recall@K:", grid.best_score_)

best_model = grid.best_estimator_


Fitting 1 folds for each of 1 candidates, totalling 1 fits


[CV] END .k_eval=1000, n_neighbors=100, weight_type=rel_freq; total time=13.4min
Best params: {'k_eval': 1000, 'n_neighbors': 100, 'weight_type': 'rel_freq'}
Best mean Recall@K: 0.4253046834201508


## Hàm gợi ý top-K item cho 1 user cụ thể

In [16]:
def recommend_for_user(model: ItemItemCFStage1, user_id, top_k=100):
    """
    Gợi ý top_k item_id cho 1 user_id (id gốc).
    Sử dụng model đã được fit (best_model).
    """
    # Map user_id → user_idx (index nội bộ)
    user_idx = model.user_id_to_index_.get(user_id, None)
    if user_idx is None:
        # cold-start user: hiện Stage 1 này không xử lý
        # có thể return danh sách item phổ biến ở ngoài
        return []

    item_idx_list = model._recommend_for_user_index(user_idx, top_k=top_k)
    item_ids = [model.item_index_to_id_[idx] for idx in item_idx_list]
    return item_ids

# Ví dụ test nhanh:
if len(df_train) > 0:
    some_user = df_train["customer_id"].iloc[2]
    candidates = recommend_for_user(best_model, some_user, top_k=10)
    print("User:", some_user)
    print("Top-10 candidate:", candidates)


User: 512190
Top-10 candidate: ['0832019490002', '4783000000009', '2808000000001', '0085000000071', '4783000000002', '2016000000002', '4422000000002', '1606000000012', '2803000000013', '2803000000011']


In [17]:
import joblib
# best_model: kết quả tốt nhất từ Stage 1 (ItemItemCFStage1)
joblib.dump(best_model, "stage1_item_item_cf.pkl")

['stage1_item_item_cf.pkl']

# Stage 2

In [18]:
import numpy as np

def precision_at_k(pred, gt, hist, filter_bought_items=True, K=10): # prediction, ground-truth, history items, candidate items
    precisions = []
    ideal_precs = []
    ncold_start = 0
    cold_start_users = []
    nusers = len(gt.keys())
    for user in gt.keys():
        if (user not in hist) or (user not in pred):
            ncold_start += 1
            cold_start_users.append(user) # THINKING: để giảm cold start có thể tăng khoảng HISTORY
            continue
        gt_items = gt[user]
        relevant_items = set(gt_items)
        if filter_bought_items:
            relevant_items -= set(hist[user])
        # Compute precision@k
        hits = len(set(pred[user][:K]) & relevant_items)
        precisions.append(hits / K)
    return np.mean(precisions), cold_start_users


## Import, load ground truth, chuẩn bị lịch sử (hist) cho Stage 2

In [21]:
import pickle
import json
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# 1) Load ground truth tháng 01/2025
gt_raw = pd.read_pickle("groundtruth.pkl")

# Giả định gt_raw là dict: {user_id: [item1, item2, ...]}
if isinstance(gt_raw, dict):
    gt = gt_raw
elif isinstance(gt_raw, pd.Series):
    gt = gt_raw.to_dict()
else:
    # Nếu format khác, bạn in ra để chỉnh tay:
    print("groundtruth.pkl format:", type(gt_raw))
    # TODO: chỉnh parse cho đúng cấu trúc của bạn
    raise ValueError("Không biết format groundtruth.pkl, cần chỉnh lại parsing.")

# 2) Chuẩn bị lịch sử mua trước 2025-01-01 (hist) từ df_transaction (Polars)
#    đây là lịch sử để filter_bought_items trong precision_at_k

cutoff_test = pd.Timestamp("2025-01-01")

df_trx_hist = (
    df_transaction
    .select(["customer_id", "item_id", "created_date"])
    .drop_nulls(["customer_id", "item_id", "created_date"])
    .with_columns(
        pl.col("created_date").cast(pl.Datetime).alias("created_datetime")
    )
    .filter(pl.col("created_datetime") < cutoff_test)
)

df_trx_hist_pd = df_trx_hist.to_pandas()

hist = (
    df_trx_hist_pd
    .groupby("customer_id")["item_id"]
    .apply(lambda s: list(set(s.tolist())))
    .to_dict()
)

print("Số user trong ground truth:", len(gt))
print("Số user có lịch sử trước 2025-01-01:", len(hist))

Số user trong ground truth: 391900
Số user có lịch sử trước 2025-01-01: 2442306


## Sinh candidate từ Stage 1 và lưu xuống file

In [22]:
best_model = joblib.load("stage1_item_item_cf.pkl")

K_cand = 1000  # số candidate Stage 1 per user cho Stage 2 (bạn có thể đổi 100/300/... tùy ý)

stage1_candidates = {}

for user_id in tqdm(gt.keys(), desc="Generate Stage1 candidates"):
    cand_items = best_model.recommend_for_user_id(user_id, top_k=K_cand)
    stage1_candidates[user_id] = cand_items

# Lưu lại để reuse
with open("stage1_candidates.pkl", "wb") as f:
    pickle.dump(stage1_candidates, f)

print("Số user có candidate:", len(stage1_candidates))


Generate Stage1 candidates: 100%|██████████| 391900/391900 [07:08<00:00, 915.33it/s] 


Số user có candidate: 391900


## Xây tập dữ liệu ranking cho LightGBM (Stage 2)

Chuẩn bị user/item feature

In [23]:
# Chuyển df_user, df_item sang pandas
df_user_pd = df_user.to_pandas()
df_item_pd = df_item.to_pandas()

# Rút gọn các cột cần dùng (có thể thêm/bớt tùy bạn)
user_feat_cols = [
    "customer_id",
    "gender",
    "province",
    "membership",
    "user_age_days",
    "days_since_install",
    "days_since_last_sync",
]

df_user_feat = df_user_pd[user_feat_cols].drop_duplicates(subset=["customer_id"])

item_feat_cols = [
    "item_id",
    "price",
    "category_l1",
    "category_l2",
    "brand",
    "age_group_final",
    # có thể thêm: "item_type", "gender_target_final"
]

df_item_feat = df_item_pd[item_feat_cols].drop_duplicates(subset=["item_id"])


Build DataFrame ranking

In [24]:
rows = []

for user_id, cand_items in tqdm(stage1_candidates.items(), desc="Build ranking rows"):
    gt_items = set(gt.get(user_id, []))

    for rank_pos, item_id in enumerate(cand_items):
        label = 1 if item_id in gt_items else 0
        rows.append(
            {
                "customer_id": user_id,
                "item_id": item_id,
                "label": label,
                "stage1_rank": rank_pos,  # vị trí trong output Stage1
            }
        )

df_rank = pd.DataFrame(rows)
print("Số dòng ranking:", len(df_rank))
print(df_rank.head())


Build ranking rows: 100%|██████████| 391900/391900 [01:44<00:00, 3740.83it/s]


Số dòng ranking: 277884431
   customer_id        item_id  label  stage1_rank
0      6515994  6665000000002      0            0
1      6515994  6665000000004      0            1
2      6515994  2803000000010      0            2
3      6515994  2793000000004      0            3
4      6515994  3052000000001      0            4


Join user/item features vào df_rank

In [25]:
df_rank = df_rank.merge(df_user_feat, on="customer_id", how="left")
df_rank = df_rank.merge(df_item_feat, on="item_id", how="left")

# Một số fillna cơ bản
df_rank["price"] = df_rank["price"].fillna(0.0)
df_rank["stage1_rank"] = df_rank["stage1_rank"].fillna(K_cand).astype(int)

# With categorical columns as category dtype
cat_cols = [
    "gender",
    "province",
    "membership",
    "category_l1",
    "category_l2",
    "brand",
    "age_group_final",
]

for c in cat_cols:
    if c in df_rank.columns:
        df_rank[c] = df_rank[c].astype("category")


## Tách train/valid cho LightGBM, train bằng GPU và lưu model

In [32]:
# Lấy danh sách user có label trong ranking
users_all = df_rank["customer_id"].unique()

train_users, valid_users = train_test_split(
    users_all, test_size=0.2, random_state=42
)

# df_train_rank = df_rank[df_rank["customer_id"].isin(train_users)].reset_index(drop=True)
# df_valid_rank = df_rank[df_rank["customer_id"].isin(valid_users)].reset_index(drop=True)

# print("Train users:", len(train_users), "Valid users:", len(valid_users))
# print("Train rows:", len(df_train_rank), "Valid rows:", len(df_valid_rank))


In [33]:
# 1) Xác định lại list feature
feature_cols = [
    c for c in df_rank.columns
    if c not in ["label", "customer_id", "item_id"]
]

# 2) Khai báo các cột categorical theo tên (nếu tồn tại trong df_rank)
cat_cols = [
    "gender",
    "province",
    "membership",
    "category_l1",
    "category_l2",
    "brand",
    "age_group_final",
]

cat_feature_names = [c for c in cat_cols if c in feature_cols]

# 3) Xử lý cột categorical: fill NA bằng '__MISSING__' rồi cast sang category
for c in cat_feature_names:
    # chuyển sang string, fill missing, sau đó cast về category
    df_rank[c] = df_rank[c].astype("string").fillna("__MISSING__").astype("category")

# 4) Các cột numeric: ép về số, fillna(0.0)
numeric_cols = [col for col in feature_cols if col not in cat_feature_names]

for col in numeric_cols:
    df_rank[col] = pd.to_numeric(df_rank[col], errors="coerce")
    df_rank[col] = df_rank[col].fillna(0.0)

# Không dùng df_rank.fillna(0.0) toàn bảng nữa!
print(df_rank[feature_cols].dtypes)

# 6) Tách lại train/valid như trước (nếu bạn đã tách rồi, chỉ cần update df_train_rank, df_valid_rank)
df_train_rank = df_rank[df_rank["customer_id"].isin(train_users)].reset_index(drop=True)
df_valid_rank = df_rank[df_rank["customer_id"].isin(valid_users)].reset_index(drop=True)

X_train = df_train_rank[feature_cols]
y_train = df_train_rank["label"]

X_valid = df_valid_rank[feature_cols]
y_valid = df_valid_rank["label"]

# 7) Tạo danh sách index cho categorical_feature
cat_feature_indices = [feature_cols.index(c) for c in cat_feature_names]

train_data = lgb.Dataset(
    X_train,
    label=y_train,
    categorical_feature=cat_feature_indices,
    free_raw_data=False,
)

valid_data = lgb.Dataset(
    X_valid,
    label=y_valid,
    categorical_feature=cat_feature_indices,
    free_raw_data=False,
)

print("Train users:", len(train_users), "Valid users:", len(valid_users))
print("Train rows:", len(df_train_rank), "Valid rows:", len(df_valid_rank))


stage1_rank                int64
gender                  category
province                category
membership              category
user_age_days            float64
days_since_install       float64
days_since_last_sync     float64
price                    float64
category_l1             category
category_l2             category
brand                   category
age_group_final         category
dtype: object
Train users: 313520 Valid users: 78380
Train rows: 222202093 Valid rows: 55682338


In [ ]:
params = {
    "objective": "binary",
    "metric": ["auc", "binary_logloss"],
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "min_data_in_leaf": 50,
    "verbosity": -1,
    # GPU
    "device": "gpu",      # bản mới dùng "device_type"
    "gpu_device_id": 7,      # nếu cần chỉ định GPU ID
}

evals_result = {}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.record_evaluation(evals_result),
    lgb.log_evaluation(period=50),
]

bst = lgb.train(
    params,
    train_data,
    num_boost_round=150,
    valid_sets=[train_data, valid_data],
    valid_names=["train", "valid"],
    callbacks=callbacks,
)

bst.save_model("lgb_stage2_ranking.txt", num_iteration=bst.best_iteration)

print("Best iteration:", bst.best_iteration)

In [ ]:
K_eval_final = 10  # K cho precision@K

pred = {}

for user_id, cand_items in tqdm(stage1_candidates.items(), desc="Predict Stage2 scores"):
    if len(cand_items) == 0:
        continue

    # Subset candidate rows cho user này từ df_rank
    # (để đảm bảo feature engineering giống lúc train)
    mask = (df_rank["customer_id"] == user_id) & (df_rank["item_id"].isin(cand_items))
    df_user_cand = df_rank.loc[mask, ["customer_id", "item_id"] + feature_cols].copy()

    if df_user_cand.empty:
        continue

    X_user = df_user_cand[feature_cols]
    scores = bst.predict(X_user, num_iteration=bst.best_iteration)

    df_user_cand["score"] = scores

    # Sort theo score giảm dần, lấy top K_eval_final
    df_user_cand_sorted = df_user_cand.sort_values("score", ascending=False)
    top_items = df_user_cand_sorted["item_id"].tolist()[:K_eval_final]

    pred[user_id] = top_items

# Tính precision@K theo hàm bạn cung cấp
prec, cold_users = precision_at_k(
    pred=pred,
    gt=gt,
    hist=hist,
    filter_bought_items=True,
    K=K_eval_final,
)

print(f"Precision@{K_eval_final}: {prec:.4f}")
print("Số user bị xem là cold-start theo hàm precision:", len(cold_users))
